# MOSS-Image-SFT 推論テスト（bf16 on T4）

Kaggle Dataset `masashiki/moss-image-sft`（HF の `OpenMOSS-Team/MOSS-Image-SFT` のミラー）を使って、bf16 で画像を1枚生成します。

## 注意点
- モデル全体は 27.2 GB あり、T4 の VRAM（16 GB）には載りません。そのため `enable_sequential_cpu_offload()` でモジュール単位の CPU オフロードを使います
- bf16 は T4（Turing）の Tensor Core ではネイティブサポートされません。エラーが出る場合は fp16 への切り替えを検討します
- まずは 20 steps・1024×1024 のスモークテストです。品質を上げるには steps を 50 に増やしてください

In [ ]:
import os

print("/kaggle/input exists:", os.path.exists("/kaggle/input"))
if os.path.exists("/kaggle/input"):
    print("/kaggle/input:", os.listdir("/kaggle/input"))
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, "dirs:", dirs, "files:", files[:5])
        if root.count("/") >= 5:
            break

In [ ]:
import subprocess

subprocess.run(["pip", "install", "-q", "diffusers>=0.38", "accelerate>=1.10"], check=True)

In [ ]:
import torch
from diffusers import DiffusionPipeline

MODEL_DIR = "/kaggle/input/datasets/masashiki/moss-image-sft"

pipe = DiffusionPipeline.from_pretrained(
    MODEL_DIR,
    custom_pipeline="pipeline_moss_image",
    trust_remote_code=True,
    torch_dtype={"default": torch.bfloat16, "vae": torch.float32},
)
pipe.scheduler = type(pipe.scheduler).from_config(pipe.scheduler.config)
print("pipeline loaded")

In [ ]:
pipe.enable_sequential_cpu_offload()
print("sequential cpu offload enabled")

In [ ]:
image = pipe(
    "A wide view of a refrigerator beside a kitchen counter with a fruit bowl.",
    height=1024,
    width=1024,
    num_inference_steps=20,
    guidance_scale=3.0,
    uncond_skip_layers=[22],
    generator=torch.Generator("cpu").manual_seed(42),
).images[0]
print("image generated")

In [ ]:
output_path = "/kaggle/working/moss_image_sft_test.png"
image.save(output_path)
print("saved to", output_path)